# 04 – Fetal-head detection (Phase 2, M4)
YOLO11n (COCO-pretrained) fine-tuned as a single-class detector on the HC18 split from notebook 03.
Steps: train → confidence/NMS sweep on VAL (operating point: max F1 s.t. recall ≥ 0.98) → TEST evaluation with
FP taxonomy, recall by size, galleries and a demo video. Acceptance: mAP50 ≥ 0.95 and recall ≥ 0.98 on TEST.

## 0. Setup (Colab)
Clone the repo, install the pinned requirements, point `FETALVISION_DATA` at the data folder (Google Drive or a local copy) and log in to W&B (optional – offline mode is used when no key is present).

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !git clone https://github.com/lamooooo1418/FetalVision.git  # TODO: replace with the team repository URL
    %cd FetalVision
    !pip install -q -r requirements.txt && pip install -q -e .
    from google.colab import drive; drive.mount("/content/drive")
    os.environ["FETALVISION_DATA"] = "/content/drive/MyDrive/FetalVision_data"   # must contain hc18/raw/training_set.zip + CSV
else:
    os.environ.setdefault("FETALVISION_DATA", os.path.abspath("../data"))
    %cd ..  # notebook lives in notebooks/, CLIs run from the repo root
print("FETALVISION_DATA =", os.environ["FETALVISION_DATA"])

In [ ]:
# W&B login (optional). Without a key the CLIs log offline; sync later with `wandb sync wandb/offline-run-*`.
import os
if os.environ.get("WANDB_API_KEY"):
    import wandb; wandb.login()
else:
    os.environ["WANDB_MODE"] = "offline"
    print("WANDB_MODE=offline")

In [ ]:
!python -m fetalvision.detection.train --config configs/phase2_detection.yaml

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/detection/figures/learning_curves.png']], [])):
    print(f); display(Image(filename=f, width=900))

In [ ]:
!python -m fetalvision.detection.sweep --config configs/phase2_detection.yaml

In [ ]:
import json; print(json.dumps(json.load(open("results/phase2/detection/operating_point.json")), indent=1))

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/detection/figures/sweep_prf_vs_conf.png']], [])):
    print(f); display(Image(filename=f, width=900))

In [ ]:
!python -m fetalvision.detection.evaluate --config configs/phase2_detection.yaml

In [ ]:
import json; m = json.load(open("results/phase2/detection/metrics.json"))["metrics"]
print(json.dumps({k: m[k] for k in ["test_at_operating_point", "fp_breakdown", "ultralytics_val_standard", "independent_map", "acceptance"]}, indent=1))
print(json.dumps(m["recall_by_gt_size"]["area_fraction_tertiles"], indent=1))

In [ ]:
from IPython.display import Image, display, JSON
import glob, json
for f in sorted(sum([glob.glob(g) for g in ['results/phase2/detection/figures/gallery_*.png', 'results/phase2/detection/figures/demo_gallery.png']], [])):
    print(f); display(Image(filename=f, width=900))

In [ ]:
# demo video (synthetic sequence of still test frames – HC18 has no video; see demo_video_caption.txt)
print(open("results/phase2/detection/demo_video_caption.txt").read())
from IPython.display import Video; Video("results/phase2/detection/demo_video.mp4", embed=True, width=640)